# iRivermetrics STAC Compatibility Example

This notebook demonstrates how to leverage STAC (SpatioTemporal Asset Catalog) protocols to load water mask data for use with the iRivermetrics algorithm. This integration enhances computational efficiency and scalability by allowing seamless access to large-scale geospatial datasets from cloud sources like the Digital Earth Australia (DEA) catalogue.

## 1. Setup and Imports

First, we'll import the necessary libraries.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import pystac_client
import odc.stac
import geopandas as gpd

import sys
# Adjust this path if your iRivermetrics repository is located elsewhere
sys.path.append(r'D:\00_GitHub\irivermetrics') 

from src import irm_main as irm
from src.utils import wd_batch, calc_metrics

import dask.array as da
import rioxarray as rxr
import xarray as xr
import pandas as pd
import numpy as np

import time
import random # For generating random dates for demonstration

## 2. Configure STAC Environment

To access DEA datasets stored in AWS from outside the DEA sandbox, AWS signed requests must be disabled. We also define the STAC catalog URL.

In [ ]:
# Disable AWS signed requests when accessing DEA from external environments
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

# Set the STAC catalog URL (Digital Earth Australia Sandbox)
url = 'https://explorer.sandbox.dea.ga.gov.au/stac'
catalog = pystac_client.Client.open(url)

print(f"Connected to STAC catalog: {url}")

## 3. Helper Functions for STAC Data Loading and Preprocessing

These functions encapsulate the logic for querying the STAC catalog and preprocessing the water mask data from Water Observations from Space (WoFS).

In [ ]:
from shapely.geometry import mapping

def generate_random_dates(start_date, end_date):
    """
    Generates a random start and end date within a specified range,
    ensuring the period is at least 180 days long.
    """
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    latest_start_date = end - pd.Timedelta(days=180)   
    
    random_start_date = random.choice(pd.date_range(start, latest_start_date))
    random_end_date = random_start_date + pd.Timedelta(days=random.randint(180, (end - random_start_date).days))
    
    return random_start_date.strftime('%Y-%m-%d'), random_end_date.strftime('%Y-%m-%d')

def generate_inputs_for_DEA(rcor_extent_gdf, start_date_overall, end_date_overall):
    """
    Prepares inputs (bbox, CRS, random dates) for querying DEA STAC.

    Args:
        rcor_extent_gdf (geopandas.GeoDataFrame): GeoDataFrame defining the Area of Interest.
        start_date_overall (str): Overall start date for the STAC query (e.g., '1986-01-01').
        end_date_overall (str): Overall end date for the STAC query (e.g., '2024-01-01').

    Returns:
        tuple: (bbox, crs_stac, start_date_random, end_date_random)
    """
    # Estimate UTM CRS for consistency with iRivermetrics
    crs_stac = rcor_extent_gdf.estimate_utm_crs().to_string()
    
    # Generate random dates for a specific query window
    start_date_random, end_date_random = generate_random_dates(start_date_overall, end_date_overall)
    
    if rcor_extent_gdf.crs.to_string() != 'EPSG:4326':
        rcor_extent_gdf = rcor_extent_gdf.to_crs(epsg=4326)
        
    # Retrieve bounding box for selected buffers    
    bbox = rcor_extent_gdf.geometry.bounds.values.flatten().tolist()
    
    return bbox, crs_stac, start_date_random, end_date_random

def load_STAC_data(bbox, catalog, collections, resolution, bands, crs_stac, start_date, end_date, max_cloud=25, max_retries=3, retry_delay=10):
    """
    Loads STAC data (e.g., WoFS water masks) from the specified catalog.

    Args:
        bbox (list): Bounding box [minx, miny, maxx, maxy].
        catalog (pystac_client.Client): Connected STAC catalog.
        collections (list): List of STAC collection IDs (e.g., ['ga_ls_wo_3']).
        resolution (int): Spatial resolution in meters (e.g., 30).
        bands (list): List of bands to load (e.g., ['water']).
        crs_stac (str): Target CRS for the loaded data.
        start_date (str): Start date for the query (YYYY-MM-DD).
        end_date (str): End date for the query (YYYY-MM-DD).
        max_cloud (int, optional): Maximum cloud percentage for filtering. Defaults to 25.
        max_retries (int, optional): Number of retries for failed STAC queries. Defaults to 3.
        retry_delay (int, optional): Delay between retries in seconds. Defaults to 10.

    Returns:
        xarray.Dataset: Loaded STAC data as an xarray Dataset.
    
    Raises:
        Exception: If all attempts to load STAC data fail.
    """
    attempts = 0
    while attempts < max_retries:
        try:
            print(f'Attempt {attempts + 1}/{max_retries}: Aquiring STAC data for {start_date} to {end_date}...')
            # Build a query with the set parameters
            query = catalog.search(
                bbox=bbox, 
                collections=collections, 
                datetime=f"{start_date}/{end_date}",
                query={"eo:cloud_cover": {"lt": max_cloud}} # Example of filtering by cloud cover
            )

            # Fetch the items
            items = list(query.items())
            
            if not items:
                print(f"No STAC items found for the given criteria. Skipping this period.")
                return None

            ds = odc.stac.load(items,
                               bands=bands,
                               crs=crs_stac,
                               resolution=resolution,
                               chunks={}, # Enable Dask lazy loading
                               groupby='solar_day',
                               bbox=bbox
                            )
            print('STAC loaded successfully.')
            return ds

        except Exception as e:
            attempts += 1
            print(f'Attempt {attempts} failed with error: {e}')
            if attempts < max_retries:
                print(f'Retrying in {retry_delay} seconds...')
                time.sleep(retry_delay)

    raise Exception('All attempts to load STAC data failed. Try again later.')

def pre_process_wofs_data(ds):
    """
    Preprocesses raw WoFS data to create a clean binary water mask (da_wmask).
    This function applies quality control measures as described in the report.

    Args:
        ds (xarray.Dataset): Raw WoFS data loaded from STAC.

    Returns:
        xarray.DataArray: Processed binary water mask (da_wmask).
    """
    print("Preprocessing WoFS data...")
    # Step 1: Exclude any time step that is entirely NaN
    ds_clean = ds.dropna(dim='time', how='all')
    if ds_clean.sizes['time'] == 0:
        print("No valid time steps after dropping all-NaN. Returning None.")
        return None

    crs = ds_clean.rio.crs

    # Step 2: Compute a mask for timesteps that do not contain the value 2 (non-water/no data for some WoFS products)
    # This specifically removes time steps that have no 'water' pixels (value 2 for WoFS) across the spatial extent
    # NOTE: This assumes WoFS 'water' band uses value 2 for water. Always check WoFS product documentation.
    mask_with_water = (ds_clean['water'] == 2).any(dim=['y', 'x'])
    ds_clean = ds_clean.sel(time=mask_with_water)
    
    if ds_clean.sizes['time'] == 0:
        print("No valid time steps after filtering for water pixels. Returning None.")
        return None

    # Step 3: Change values where water == 128 (wet) to 1 (water)
    da_wmask = xr.where(ds_clean['water'] == 128, 1, ds_clean['water'])
    
    # Step 4: Set all values that are not 0 (no water) or 1 (water) to -1 (no data)
    da_wmask = xr.where((da_wmask != 0) & (da_wmask != 1), -1, da_wmask)
    
    # Step 5: Set the attribute _FillValue to -1
    da_wmask.attrs['_FillValue'] = -1
    da_wmask.rio.write_crs(crs, inplace=True)

    # Apply quality control based on valid pixel percentage
    initial_valid_pixels = (da_wmask != -1).sum(dim=['y', 'x'])
    total_pixels = da_wmask.shape[1] * da_wmask.shape[2]
    valid_percentage = (initial_valid_pixels / total_pixels) * 100

    # Exclude timesteps with less than 70% valid data initially
    da_wmask = da_wmask.isel(time=(valid_percentage >= 70))
    if da_wmask.sizes['time'] == 0:
        print("No valid time steps after 70% valid data threshold. Returning None.")
        return None
        
    # Forward fill missing values over time
    da_wmask = da_wmask.ffill(dim='time', limit=2) # Limit to 2 observations
    # Backward fill for remaining missing values at the end of the time series
    da_wmask = da_wmask.bfill(dim='time', limit=2) # Propagate up to two observations

    # Re-evaluate and exclude any time steps with less than 95% valid pixels
    final_valid_pixels = (da_wmask != -1).sum(dim=['y', 'x'])
    final_valid_percentage = (final_valid_pixels / total_pixels) * 100
    da_wmask = da_wmask.isel(time=(final_valid_percentage >= 95))
    
    if da_wmask.sizes['time'] == 0:
        print("No valid time steps after 95% valid data threshold. Returning None.")
        return None

    print(f"Preprocessing complete. {da_wmask.sizes['time']} valid time steps will be used to calculate metrics.")
    return da_wmask


## 4. Define Study Area and iRivermetrics Parameters

We'll define the path to our river section shapefile (e.g., Gilbert River ), output directories, and other parameters for iRivermetrics and STAC queries.

In [ ]:
# --- Gilbert River Example ---
rivers_path_gilbert = r'D:\2.1_WPE_waterholes\test_efficiency\Gilbert\shp\Gilbert_river_buffer_1000.shp'
path_outputs_gilbert = r'D:\2.1_WPE_waterholes\test_efficiency\Gilbert\results_stac_example'

# Overall date range for the STAC query
start_date_overall = '1986-01-01'
end_date_overall = '2024-01-01'

collections = ['ga_ls_wo_3'] # WoFS Collection 3
max_cloud = 10 # Maximum cloud cover percentage for filtering STAC items
bands = ['water']
resolution = 30 # Spatial resolution in meters

# Load the river corridor extent (Area of Interest)
rcor_extent_gilbert = gpd.read_file(rivers_path_gilbert)

# For demonstration, let's select a single random section if the shapefile contains multiple,
# or use the entire loaded shapefile if it's a single feature.
# Here we'll take the first feature for simplicity.
if not rcor_extent_gilbert.empty:
    rcor_extent_single = rcor_extent_gilbert.iloc[[0]] 
    river_name_single = rcor_extent_single['NAME'].iloc[0].split()[0] if 'NAME' in rcor_extent_single.columns else "Gilbert_Section"
else:
    raise ValueError("No river sections found in the provided shapefile.")

print(f"Processing river section: {river_name_single}")

## 5. Concrete STAC Query and iRivermetrics Integration Example

This section demonstrates the core workflow:
1. Prepare STAC query inputs based on the river section.
2. Load the water mask data from the STAC catalog using `odc.stac.load`.
3. Preprocess the loaded data into the `da_wmask` format expected by iRivermetrics.
4. Run the `calculate_metrics` module of iRivermetrics.

In [ ]:
from IPython.display import clear_output

# Prepare inputs for the STAC query using the selected river section
bbox, crs_stac, start_date_query, end_date_query = generate_inputs_for_DEA(rcor_extent_single, start_date_overall, end_date_overall)

# Define the output directory for this specific river section
outdir_single_section = os.path.join(path_outputs_gilbert, f'results_iRiverMetrics_{river_name_single}')
os.makedirs(outdir_single_section, exist_ok=True) # Create the output directory if it doesn't exist

print(f"STAC Query Parameters:")
print(f"  BBox: {bbox}")
print(f"  CRS: {crs_stac}")
print(f"  Date Range: {start_date_query} to {end_date_query}")
print(f"  Collections: {collections}")
print(f"  Max Cloud: {max_cloud}%")
print(f"  Resolution: {resolution}m")
print(f"  Output Directory: {outdir_single_section}")

# 1. Load data from STAC
# The `load_STAC_data` function uses odc.stac.load internally,
# which handles Dask integration for lazy loading of large datasets.
raw_stac_data = load_STAC_data(bbox, catalog, collections, resolution, bands, crs_stac, start_date_query, end_date_query, max_cloud=max_cloud)

if raw_stac_data is not None:
    # Display information about the loaded Dask Xarray DataArray
    print("\nLoaded STAC data (raw):")
    print(raw_stac_data)

    # 2. Preprocess the WoFS data for iRivermetrics
    da_wmask = pre_process_wofs_data(raw_stac_data)

    if da_wmask is not None:
        print("\nProcessed Water Mask (da_wmask):")
        print(da_wmask)
        
        # 3. Run iRivermetrics calculate_metrics module
        print("\nCalculating metrics with iRivermetrics...")
        # The `calculate_metrics` function will leverage Dask for parallel processing
        # if the input `da_wmask` is a Dask array, which it is here.
        metrics_results = irm.calculate_metrics(da_wmask, rcor_extent_single, outdir_single_section, export_shp=False)
        
        print("\nMetrics calculation complete. Results saved to:")
        print(os.path.join(outdir_single_section, 'results_iRiverMetrics', 'metrics')) # Standard output path structure
        print("\nFirst few rows of calculated metrics (if available):")
        if isinstance(metrics_results, pd.DataFrame) and not metrics_results.empty:
            print(metrics_results.head())
        else:
            print("No DataFrame returned or it is empty.")
    else:
        print("Skipping metrics calculation due to empty preprocessed water mask.")
else:
    print("Skipping preprocessing and metrics calculation due to no raw STAC data.")

clear_output(wait=True)
print("Demonstration of STAC compatibility complete for one section.")

## 6. Exporting Processed Water Masks (Optional)

This section demonstrates how you might export the preprocessed water masks to individual GeoTIFF files, which can be useful for visual inspection or further analysis.

In [ ]:
# --- OPTIONAL: Exporting Valid WoFS Images ---
# This part is for demonstration if you want to save the preprocessed water masks locally.
# It can be resource-intensive for large datasets.

if da_wmask is not None:
    output_dir_tifs = os.path.join(path_outputs_gilbert, 'wofs_processed_tifs')
    os.makedirs(output_dir_tifs, exist_ok=True)
    print(f"\nExporting {da_wmask.sizes['time']} processed water masks to: {output_dir_tifs}")

    for i in range(da_wmask.sizes['time']):
        try:
            date_str = da_wmask.time[i].dt.strftime('%Y-%m-%d').values
            output_filename = os.path.join(output_dir_tifs, f'{date_str}.tif')
            
            # Ensure the output data type is compatible (e.g., int8 for binary masks)
            da_wmask.isel(time=i).astype(np.int8).rio.to_raster(output_filename)
            # print(f"Exported: {output_filename}") # Uncomment for verbose output
        except Exception as e:
            print(f"Error exporting timestep {date_str}: {e}")
    print("Export of processed water masks complete.")
else:
    print("\nNo preprocessed water mask (da_wmask) to export.")